In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Arial"]
rcParams["font.size"] = 10
rcParams["axes.linewidth"] = 0.8
rcParams["xtick.major.width"] = 0.8
rcParams["ytick.major.width"] = 0.8
rcParams["xtick.major.size"] = 3
rcParams["ytick.major.size"] = 3
rcParams["pdf.fonttype"] = 42
rcParams["ps.fonttype"] = 42

# =========================

# =========================
DEG_FILE = "./edgeR_QLF_by_celltype_blockReplicate.csv"

OUT_DIR = "./Volcano_LABEL_ONLY"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================

# =========================
FDR_CUTOFF = 0.05
LOGFC_CUTOFF = 0.25

FIG_W, FIG_H = 7, 6
DPI = 300

# =========================

# =========================
GENES_TO_LABEL = [
    "Fam50a",
   
]


GENES_TO_LABEL = set([g.strip().upper() for g in GENES_TO_LABEL if g.strip() != ""])


# =========================

def safe_name(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(x))


def volcano_plot_label_only(
    df: pd.DataFrame,
    out_prefix: str,
    title: str,
    genes_to_label: set,
    fdr_cutoff=0.05,
    logfc_cutoff=0.25
):
    dd = df.copy()

    # 找 p 值列
    if "PValue" in dd.columns:
        pcol = "PValue"
    elif "pvalue" in dd.columns:
        pcol = "pvalue"
    else:
        raise ValueError("缺少 PValue 列")

    if "FDR" not in dd.columns:
        raise ValueError("缺少 FDR 列")

    dd["gene"] = dd["gene"].astype(str).str.strip()
    dd["gene_up"] = dd["gene"].str.upper()

    dd["logFC"] = pd.to_numeric(dd["logFC"], errors="coerce")
    dd["FDR"]   = pd.to_numeric(dd["FDR"], errors="coerce")
    dd[pcol]    = pd.to_numeric(dd[pcol], errors="coerce")
    dd = dd.dropna(subset=["logFC", "FDR", pcol]).copy()

    # -log10(p)
    dd["neglog10P"] = -np.log10(dd[pcol].clip(lower=1e-300))

    
    dd["sig_class"] = "NS"
    dd.loc[(dd["FDR"] < fdr_cutoff) & (dd["logFC"] >= logfc_cutoff), "sig_class"] = "UP"
    dd.loc[(dd["FDR"] < fdr_cutoff) & (dd["logFC"] <= -logfc_cutoff), "sig_class"] = "DOWN"

    color_map = {"NS": "#BDBDBD", "UP": "#E64B35", "DOWN": "#4DBBD5"}

    plt.figure(figsize=(FIG_W, FIG_H))

    
    for cls in ["NS", "UP", "DOWN"]:
        sub = dd[dd["sig_class"] == cls]
        plt.scatter(
            sub["logFC"],
            sub["neglog10P"],
            s=10 if cls == "NS" else 14,
            c=color_map[cls],
            alpha=0.8 if cls != "NS" else 0.45,
            linewidths=0,
            label=cls
        )

    
    plt.axvline(logfc_cutoff,  color="black", linewidth=1, linestyle="--")
    plt.axvline(-logfc_cutoff, color="black", linewidth=1, linestyle="--")
    plt.axhline(-np.log10(fdr_cutoff), color="black", linewidth=1, linestyle=":")

    plt.title(title)
    plt.xlabel("log2 Fold Change (logFC)")
    plt.ylabel("-log10(PValue)")
    plt.legend(frameon=False, loc="upper right")

    # =========================
    
    # =========================
    lab = dd[dd["gene_up"].isin(genes_to_label)].copy()

    
    if lab.shape[0] == 0:
        plt.tight_layout()
        plt.savefig(out_prefix + ".pdf")
        plt.savefig(out_prefix + ".png", dpi=DPI)
        plt.close()
        return

    
    lab = lab.sort_values("FDR").copy()
    for i, (_, r) in enumerate(lab.iterrows()):
        g = str(r["gene"])
        x = float(r["logFC"])
        y = float(r["neglog10P"])

        plt.text(
            x,
            y + (i % 5) * 0.05,    
            g,
            fontsize=9,
            ha="left" if x >= 0 else "right",
            va="bottom"
        )

    plt.tight_layout()

    
    plt.savefig(out_prefix + ".pdf")
    plt.savefig(out_prefix + ".png", dpi=DPI)
    plt.close()


# =========================

# =========================
deg = pd.read_csv(DEG_FILE)
deg["gene"] = deg["gene"].astype(str).str.strip()

required = {"gene", "celltype_coarse", "contrast", "logFC", "PValue", "FDR"}
missing = required - set(deg.columns)
if missing:
    raise ValueError(f"❌ DEG 文件缺少列: {missing}")

groups = deg[["celltype_coarse", "contrast"]].drop_duplicates()
print("Total groups:", groups.shape[0])


# =========================

# =========================
for _, g in groups.iterrows():
    ct = g["celltype_coarse"]
    co = g["contrast"]

    sub = deg[(deg["celltype_coarse"] == ct) & (deg["contrast"] == co)].copy()
    if sub.shape[0] == 0:
        continue

    safe_ct = safe_name(ct)
    safe_co = safe_name(co)

    out_prefix = os.path.join(OUT_DIR, f"{safe_ct}__{safe_co}__volcano_labelOnly")
    title = f"{ct} | {co}\nlabelOnly ({len(GENES_TO_LABEL)} genes)"

    volcano_plot_label_only(
        sub,
        out_prefix=out_prefix,
        title=title,
        genes_to_label=GENES_TO_LABEL,
        fdr_cutoff=FDR_CUTOFF,
        logfc_cutoff=LOGFC_CUTOFF
    )

    print("✅ Volcano saved:", out_prefix)

print("\nALL DONE. Output folder:")
print(OUT_DIR)


Total groups: 4
✅ Volcano saved: ./Volcano_LABEL_ONLY/Deep-layer_neurons__Fam50a_vs_NTC__volcano_labelOnly
✅ Volcano saved: ./Volcano_LABEL_ONLY/Hippocampal_neurons__Fam50a_vs_NTC__volcano_labelOnly
✅ Volcano saved: ./Volcano_LABEL_ONLY/Inhibitory_interneurons__Fam50a_vs_NTC__volcano_labelOnly
✅ Volcano saved: ./Volcano_LABEL_ONLY/Sup_-layer_neurons__Fam50a_vs_NTC__volcano_labelOnly

ALL DONE. Output folder:
./Volcano_LABEL_ONLY
